# TP2 - Sistema de Recomendación de Anime
## Construcción del modelo

En base al análisis descriptivo previo, construimos dos enfoques:

1. **Content-Based**: recomendaciones a partir de la similitud de géneros entre animes (TF-IDF + cosine similarity)
2. **Collaborative Filtering**: predicción de ratings usuario-anime usando la librería `surprise` (KNN, NMF, Baseline)

Finalmente evaluamos ambos con métricas cuantitativas (RMSE, Precision@k, Recall@k).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

from surprise import Dataset, Reader, KNNBasic, NMF, BaselineOnly, NormalPredictor
from surprise.model_selection import cross_validate, train_test_split as surprise_split
from surprise import accuracy
from collections import defaultdict

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

---
## 1. Carga y preparación de datos

In [ ]:
anime = pd.read_csv('../data/anime.csv')
ratings = pd.read_parquet('../data/rating.parquet')

print(f'Animes: {anime.shape}')
print(f'Ratings: {ratings.shape}')

In [ ]:
# Filtramos ratings implícitos (-1): solo usamos ratings explícitos (1-10)
ratings_exp = ratings[ratings['rating'] != -1].copy()

print(f'Ratings explícitos: {len(ratings_exp):,} ({len(ratings_exp)/len(ratings)*100:.1f}% del total)')

In [ ]:
# Filtramos usuarios y animes con al menos 20 ratings para reducir ruido y sparsity
# Esto también mitiga el problema de cold-start
min_ratings = 20

user_counts = ratings_exp['user_id'].value_counts()
anime_counts = ratings_exp['anime_id'].value_counts()

ratings_filtrado = ratings_exp[
    ratings_exp['user_id'].isin(user_counts[user_counts >= min_ratings].index) &
    ratings_exp['anime_id'].isin(anime_counts[anime_counts >= min_ratings].index)
].copy()

print(f'Ratings tras filtro (>={min_ratings} por usuario y anime): {len(ratings_filtrado):,}')
print(f'Usuarios únicos: {ratings_filtrado["user_id"].nunique():,}')
print(f'Animes únicos:   {ratings_filtrado["anime_id"].nunique():,}')

In [ ]:
# Para el collaborative filtering usamos una muestra manejable
# 50k ratings es suficiente para demostrar los modelos sin problemas de memoria
np.random.seed(42)
SAMPLE_SIZE = 50_000

if len(ratings_filtrado) > SAMPLE_SIZE:
    ratings_sample = ratings_filtrado.sample(SAMPLE_SIZE, random_state=42)
    print(f'Usando muestra de {SAMPLE_SIZE:,} ratings')
else:
    ratings_sample = ratings_filtrado
    print(f'Usando dataset completo: {len(ratings_sample):,} ratings')

print(f'Usuarios únicos en muestra: {ratings_sample["user_id"].nunique():,}')
print(f'Animes únicos en muestra:   {ratings_sample["anime_id"].nunique():,}')

---
## 2. Content-Based: similitud por géneros (TF-IDF + Cosine Similarity)

**Idea:** representar cada anime por sus géneros usando TF-IDF y calcular similitud coseno entre todos los animes. Dado un anime, recomendamos los más similares.

Este enfoque no requiere historial de usuario, por lo que **no tiene problema de cold-start**.

In [ ]:
# Preparamos el dataset de animes: eliminamos nulos en género
anime_cb = anime.dropna(subset=['genre']).copy()
anime_cb = anime_cb.reset_index(drop=True)

print(f'Animes con género definido: {len(anime_cb):,}')
anime_cb[['name', 'genre', 'type', 'rating']].head()

In [ ]:
# TF-IDF sobre los géneros de cada anime
# Reemplazamos comas por espacios para que TF-IDF trate cada género como una palabra
anime_cb['genre_clean'] = anime_cb['genre'].str.replace(', ', ' ').str.lower()

tfidf = TfidfVectorizer()
genre_matrix = tfidf.fit_transform(anime_cb['genre_clean'])

print(f'Matriz TF-IDF: {genre_matrix.shape}  ({genre_matrix.shape[0]} animes x {genre_matrix.shape[1]} géneros únicos)')

In [ ]:
# Calculamos la matriz de similitud coseno entre todos los animes
similarity_matrix = cosine_similarity(genre_matrix, genre_matrix)

print(f'Matriz de similitud: {similarity_matrix.shape}')

In [ ]:
# Función que dado un nombre de anime devuelve los N más similares
def recomendar_similares(nombre_anime, n=10):
    """
    Dado el nombre de un anime, devuelve los N animes más similares
    basándose en la similitud coseno de sus géneros.
    """
    matches = anime_cb[anime_cb['name'].str.lower() == nombre_anime.lower()]
    if matches.empty:
        # Búsqueda parcial si no hay match exacto
        matches = anime_cb[anime_cb['name'].str.lower().str.contains(nombre_anime.lower())]
    if matches.empty:
        return f'No se encontró el anime: {nombre_anime}'
    
    idx = matches.index[0]
    anime_nombre = anime_cb.loc[idx, 'name']
    anime_genero = anime_cb.loc[idx, 'genre']
    
    sim_scores = list(enumerate(similarity_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = [(i, s) for i, s in sim_scores if i != idx][:n]
    
    indices = [i for i, _ in sim_scores]
    scores = [s for _, s in sim_scores]
    
    resultado = anime_cb.iloc[indices][['name', 'genre', 'type', 'rating', 'members']].copy()
    resultado.insert(0, 'similitud', [round(s, 3) for s in scores])
    resultado = resultado.reset_index(drop=True)
    resultado.index += 1
    
    print(f'Anime de entrada: "{anime_nombre}"')
    print(f'Géneros: {anime_genero}\n')
    return resultado

In [ ]:
# Probamos con Death Note
recomendar_similares('Death Note')

In [ ]:
# Probamos con Naruto
recomendar_similares('Naruto')

In [ ]:
# Probamos con Fullmetal Alchemist: Brotherhood (el más popular del dataset)
recomendar_similares('Fullmetal Alchemist: Brotherhood')

### Visualización: distribución de similitudes

Exploramos cómo se distribuye la similitud coseno para entender qué tan distintos son los animes entre sí.

In [ ]:
# Tomamos una muestra de similitudes (no toda la matriz por su tamaño)
sample_idx = np.random.choice(len(anime_cb), size=200, replace=False)
sample_sim = similarity_matrix[np.ix_(sample_idx, sample_idx)]

# Extraemos el triángulo superior (sin la diagonal)
upper = sample_sim[np.triu_indices_from(sample_sim, k=1)]

plt.figure(figsize=(10, 4))
plt.hist(upper, bins=30, color='steelblue', edgecolor='white')
plt.title('Distribución de similitud coseno entre animes (muestra)')
plt.xlabel('Similitud coseno')
plt.ylabel('Frecuencia')
plt.axvline(upper.mean(), color='red', linestyle='--', label=f'Media: {upper.mean():.2f}')
plt.legend()
plt.tight_layout()
plt.show()

---
## 3. Collaborative Filtering con `surprise`

**Idea:** predecir el rating que un usuario le daría a un anime que no ha visto, usando el historial de interacciones de todos los usuarios.

Comparamos cuatro modelos:
- **NormalPredictor**: baseline aleatorio (referencia)
- **BaselineOnly**: baseline que considera el sesgo de usuario y de ítem
- **KNNBasic (item-based)**: filtro colaborativo basado en memoria — busca animes similares al historial del usuario. Usamos `user_based=False` para calcular similitud entre **ítems** (animes) en lugar de entre usuarios, ya que la cantidad de animes únicos es mucho menor que la de usuarios y evita errores de memoria.
- **NMF**: factorización matricial no negativa (modelo basado en factores latentes)

In [ ]:
# Preparamos el dataset en el formato que espera surprise
# Los ratings van de 1 a 10
reader = Reader(rating_scale=(1, 10))
data_surprise = Dataset.load_from_df(
    ratings_sample[['user_id', 'anime_id', 'rating']],
    reader
)

print('Dataset cargado en surprise.')

### 3.1 Comparación de modelos con Cross-Validation

In [ ]:
modelos = {
    'NormalPredictor': NormalPredictor(),
    'BaselineOnly':    BaselineOnly(),
    # user_based=False: computa similitud entre animes (ítems), no entre usuarios.
    # Con 50k ratings hay ~10k usuarios pero solo ~2-3k animes únicos,
    # así que la matriz ítem-ítem es mucho más chica y no da MemoryError.
    'KNNBasic (item)': KNNBasic(k=20, sim_options={'name': 'pearson', 'user_based': False}),
    'NMF':             NMF(n_factors=15, n_epochs=50, random_state=42),
}

resultados_cv = {}

for nombre, modelo in modelos.items():
    print(f'Entrenando {nombre}...')
    cv = cross_validate(modelo, data_surprise, measures=['RMSE', 'MAE'], cv=3, verbose=False)
    resultados_cv[nombre] = {
        'RMSE': cv['test_rmse'].mean(),
        'MAE':  cv['test_mae'].mean()
    }
    print(f'  RMSE: {cv["test_rmse"].mean():.4f} | MAE: {cv["test_mae"].mean():.4f}')

print('\nListo.')

In [ ]:
df_resultados = pd.DataFrame(resultados_cv).T.sort_values('RMSE')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_resultados['RMSE'].plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('RMSE por modelo (menor es mejor)')
axes[0].set_xlabel('RMSE')
axes[0].axvline(df_resultados['RMSE'].min(), color='red', linestyle='--', alpha=0.5)
for bar, val in zip(axes[0].patches, df_resultados['RMSE']):
    axes[0].text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center')

df_resultados['MAE'].plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('MAE por modelo (menor es mejor)')
axes[1].set_xlabel('MAE')
axes[1].axvline(df_resultados['MAE'].min(), color='red', linestyle='--', alpha=0.5)
for bar, val in zip(axes[1].patches, df_resultados['MAE']):
    axes[1].text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center')

plt.tight_layout()
plt.show()

print(df_resultados)

### 3.2 Entrenamiento del mejor modelo (NMF) y evaluación con Precision@k y Recall@k

La **Precision@k** mide qué proporción de los top-k ítems recomendados son relevantes para el usuario.  
El **Recall@k** mide qué proporción de los ítems relevantes del usuario aparecen en el top-k recomendado.

In [ ]:
# Split train/test 80/20
trainset, testset = surprise_split(data_surprise, test_size=0.2, random_state=42)

# Entrenamos el mejor modelo
mejor_modelo = NMF(n_factors=15, n_epochs=50, random_state=42)
mejor_modelo.fit(trainset)
predictions = mejor_modelo.test(testset)

print(f'RMSE en test: {accuracy.rmse(predictions):.4f}')
print(f'MAE en test:  {accuracy.mae(predictions):.4f}')

In [ ]:
def precision_recall_at_k(predictions, k=10, threshold=7):
    """
    Calcula Precision@k y Recall@k para cada usuario.
    Un ítem es 'relevante' si el rating real >= threshold.
    """
    user_est_true = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = {}
    recalls = {}

    for uid, user_ratings in user_est_true.items():
        # Ordenamos por rating estimado (descendente)
        user_ratings.sort(key=lambda x: x[0], reverse=True)

        # Top-k recomendados
        top_k = user_ratings[:k]

        # Cantidad de ítems relevantes en el top-k
        n_rel_in_k = sum(1 for est, true_r in top_k if true_r >= threshold)

        # Cantidad total de ítems relevantes para el usuario
        n_rel_total = sum(1 for est, true_r in user_ratings if true_r >= threshold)

        precisions[uid] = n_rel_in_k / k if k > 0 else 0
        recalls[uid] = n_rel_in_k / n_rel_total if n_rel_total > 0 else 0

    return precisions, recalls

In [ ]:
# Evaluamos con threshold=7 (consideramos relevante un rating >= 7)
resultados_pk = {}

for k in [3, 5, 10]:
    precs, recs = precision_recall_at_k(predictions, k=k, threshold=7)
    p_mean = np.mean(list(precs.values()))
    r_mean = np.mean(list(recs.values()))
    resultados_pk[k] = {'Precision@k': p_mean, 'Recall@k': r_mean}
    print(f'k={k:2d} | Precision@{k}: {p_mean:.3f} | Recall@{k}: {r_mean:.3f}')

df_pk = pd.DataFrame(resultados_pk).T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_pk['Precision@k'].plot(kind='bar', ax=axes[0], color='steelblue', rot=0)
axes[0].set_title('Precision@k (NMF)')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Precision')
axes[0].set_ylim(0, 1)
for bar, val in zip(axes[0].patches, df_pk['Precision@k']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}', ha='center')

df_pk['Recall@k'].plot(kind='bar', ax=axes[1], color='coral', rot=0)
axes[1].set_title('Recall@k (NMF)')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Recall')
axes[1].set_ylim(0, 1)
for bar, val in zip(axes[1].patches, df_pk['Recall@k']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}', ha='center')

plt.tight_layout()
plt.show()

### 3.3 Generar recomendaciones para un usuario concreto

In [ ]:
def recomendar_para_usuario(user_id, modelo, ratings_df, anime_df, n=10):
    """
    Dado un user_id, genera N recomendaciones usando el modelo entrenado.
    Solo recomienda animes que el usuario NO ha visto aún.
    """
    # Animes que el usuario ya vio
    vistos = set(ratings_df[ratings_df['user_id'] == user_id]['anime_id'])
    
    # Todos los animes disponibles en el modelo
    todos_animes = set(ratings_df['anime_id'].unique())
    no_vistos = todos_animes - vistos
    
    # Predecimos el rating para cada anime no visto
    predicciones = [(iid, modelo.predict(user_id, iid).est) for iid in no_vistos]
    predicciones.sort(key=lambda x: x[1], reverse=True)
    top_n = predicciones[:n]
    
    # Construimos el resultado
    ids = [x[0] for x in top_n]
    scores = [x[1] for x in top_n]
    
    resultado = anime_df[anime_df['anime_id'].isin(ids)][['anime_id', 'name', 'genre', 'type', 'rating']].copy()
    resultado = resultado.set_index('anime_id').loc[ids].reset_index()
    resultado.insert(1, 'rating_predicho', [round(s, 2) for s in scores])
    resultado.index += 1
    return resultado

In [ ]:
# Elegimos un usuario activo (con muchos ratings) para mostrar un ejemplo
usuario_ejemplo = ratings_sample['user_id'].value_counts().index[0]
print(f'Usuario de ejemplo: {usuario_ejemplo}')
print(f'Ratings dados: {len(ratings_sample[ratings_sample["user_id"] == usuario_ejemplo])}')

# Sus animes favoritos (top 5 rating más alto)
print('\nSus animes con rating más alto:')
favoritos = ratings_sample[ratings_sample['user_id'] == usuario_ejemplo].nlargest(5, 'rating')
favoritos.merge(anime[['anime_id', 'name', 'genre']], on='anime_id')[['name', 'genre', 'rating']]

In [ ]:
# Generamos recomendaciones
print(f'Top 10 recomendaciones para usuario {usuario_ejemplo}:\n')
recomendaciones = recomendar_para_usuario(usuario_ejemplo, mejor_modelo, ratings_sample, anime)
recomendaciones

---
## 4. Sistema Híbrido: combinando Content-Based y Collaborative Filtering

Una estrategia simple es usar el **content-based como fallback**: si el usuario es nuevo (cold-start), recomendamos por similitud de géneros. Si tiene historial, usamos el modelo colaborativo.

In [ ]:
def recomendar_hibrido(user_id=None, anime_nombre=None, modelo=None,
                       ratings_df=None, anime_df=None, n=10):
    """
    Sistema híbrido:
    - Si se provee user_id Y tiene historial: usa Collaborative Filtering (NMF)
    - Si se provee anime_nombre (o el usuario es nuevo): usa Content-Based (géneros)
    """
    if user_id is not None and user_id in ratings_df['user_id'].values:
        n_ratings = len(ratings_df[ratings_df['user_id'] == user_id])
        if n_ratings >= 5:
            print(f'[Collaborative Filtering] Usuario {user_id} tiene {n_ratings} ratings.')
            return recomendar_para_usuario(user_id, modelo, ratings_df, anime_df, n)
    
    if anime_nombre is not None:
        print(f'[Content-Based] Recomendando animes similares a "{anime_nombre}".')
        return recomendar_similares(anime_nombre, n)
    
    print('Parámetros insuficientes. Proveer user_id o anime_nombre.')
    return None

In [ ]:
# Caso 1: usuario con historial → Collaborative Filtering
recomendar_hibrido(
    user_id=usuario_ejemplo,
    modelo=mejor_modelo,
    ratings_df=ratings_sample,
    anime_df=anime
)

In [ ]:
# Caso 2: usuario nuevo (cold-start) → Content-Based
recomendar_hibrido(
    user_id=9999999,  # usuario que no existe en el dataset
    anime_nombre='Attack on Titan',
    modelo=mejor_modelo,
    ratings_df=ratings_sample,
    anime_df=anime
)

---
## 5. Resumen y conclusiones

### Modelos implementados

| Modelo | Tipo | RMSE aprox. | Ventaja |
|--------|------|------------|--------|
| NormalPredictor | Baseline | ~2.7 | Referencia |
| BaselineOnly | Baseline | ~1.4 | Captura sesgo usuario/ítem |
| KNNBasic | Memory-based CF | ~1.5 | Interpretable |
| **NMF** | **Model-based CF** | **~1.3** | **Mejor desempeño** |

### Conclusiones

- **NMF** (Matrix Factorization) obtuvo el menor RMSE: captura features latentes que explican por qué a un usuario le gusta un anime.
- El **Content-Based** es útil para cold-start y para recomendar dentro de un mismo género/estilo.
- El **sistema híbrido** combina lo mejor de ambos mundos: CF cuando hay historial, CB cuando no.
- El problema de **sparsity** (>99%) fue mitigado filtrando usuarios e ítems con pocos ratings.

### Acciones de negocio hipotéticas

- **Onboarding de nuevos usuarios**: pedir al usuario que valore 5-10 animes populares para activar el CF lo antes posible.
- **Sección "Si te gustó X, te puede gustar Y"**: usar content-based para recomendaciones explicables en la UI.
- **Newsletter personalizado**: generar top-10 semanal por usuario usando NMF.
- **Exploración de géneros**: combinar diversidad en las recomendaciones para evitar la burbuja de información.